In [ ]:
import os

# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import pandas as pd  # requires: pip install pandas
import numpy as np
import matplotlib.pyplot as plt
import torch
from chronos import BaseChronosPipeline, Chronos2Pipeline

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

## Incidence forecast

In [ ]:
data_path = "D:/Users/peter/Desktop/UROPS/Foundation models/covid_incidence.csv"
covid_df = pd.read_csv(data_path, index_col=0, parse_dates=True)

states = ["az", "ca", "il", "md", "nj", "ny"]

state_dfs = {}
for st in states:
    df_st = (
        covid_df[[st]]                             # keep one state
        .rename(columns={st: "target"})             # target column
        .rename_axis("timestamp")                   # name the index for reset
        .reset_index()                              # move index to a column
        .assign(item_id=st)                         # add item_id
        [["item_id", "timestamp", "target"]]        # order columns
    )
    # (optional) enforce dtypes
    df_st["timestamp"] = pd.to_datetime(df_st["timestamp"])
    df_st["target"] = pd.to_numeric(df_st["target"], errors="coerce")
    state_dfs[st] = df_st

az_df = state_dfs["az"]
ca_df = state_dfs["ca"]
il_df = state_dfs["il"]
md_df = state_dfs["md"]
nj_df = state_dfs["nj"]
ny_df = state_dfs["ny"]

In [ ]:
input_len    = 56
pred_len     = 14
stride       = 7
T            = len(covid_df)
n_windows = (T - input_len - pred_len) // stride + 1
quantiles = [0.05, 0.1, 0.15, 0.20, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]


def _quantile_cols(df, qlevels):
    cols = []
    for q in qlevels:
        # columns can be floats (0.1) or strings ("0.1")
        if q in df.columns:
            cols.append(q)
        elif f"{q:g}" in df.columns:
            cols.append(f"{q:g}")
        elif f"{q}" in df.columns:
            cols.append(f"{q}")
        else:
            raise KeyError(f"Quantile column {q} not found in output columns: {df.columns.tolist()}")
    return cols

per_state = {st: np.zeros((n_windows, pred_len, len(quantiles)), dtype=np.float32) for st in states}


for w in range(n_windows):
        start = w * stride
        end   = start + input_len          # exclusive slice end for context

        pred_start = covid_df.index[end]   # first day being forecast
        print(f"[{w+1:02d}/{n_windows}] prediction starts on {pred_start:%Y-%m-%d}")

        # Chronos-2 expects long format: item_id | timestamp | target
        batch = pd.concat(
            [ state_dfs[st].iloc[start:end, :] for st in states ],
            ignore_index=True
        )

        # ---- call pipeline ----
        # NOTE: Depending on your chronos version, the method name can be
        #   - pipe.predict_quantiles(...)
        #   - pipe.forecast(...)
        #   - pipe(...)
        # Replace the call below with the exact API your installation exposes.
        #
        # Expected result structure (pseudo):
        #   A mapping from item_id -> np.ndarray shape (pred_len, n_quantiles)
        #

        result = pipeline.predict_df(batch, prediction_length=pred_len, quantile_levels=quantiles)

        qcols = _quantile_cols(result, quantiles)
        # ---- parse results into buffers ----
        for st in states:
            block = result.loc[result["item_id"] == st, qcols].to_numpy(dtype=np.float32)
            # block shape should be (pred_len, len(quantiles))
            if block.shape != (pred_len, len(quantiles)):
                raise ValueError(f"Unexpected block shape for {st}: {block.shape}")
            per_state[st][w, :, :] = block

# ------------------------------------
# 5) Access your arrays
# ------------------------------------
az_arr = per_state["az"]  # shape (n_windows, 14, 10)
ca_arr = per_state["ca"]
il_arr = per_state["il"]
md_arr = per_state["md"]
nj_arr = per_state["nj"]
ny_arr = per_state["ny"]

print({k: v.shape for k, v in per_state.items()})

In [ ]:
# save forecast results
output_dir = "../../data/3_incidence predictions/chronos2"
for st in states:
    file_path = os.path.join(output_dir, f"{st}_forecasts.npy")
    np.save(file_path, per_state[st])